# Deep Q-Network — лаб-тетрадь

**Статья:** Mnih et al. — *Playing Atari with Deep Reinforcement Learning* (NIPS Deep Learning Workshop 2013, arXiv:1312.5602). Сюда же — приём из *Human-level control through deep RL* (Nature 2015), см. ниже про target-сеть.

🌳 **Узел дерева:** Reinforcement learning → value-based → approximate Q-learning / DQN.

**Что воспроизводим:** обучаем Q-сеть с experience replay и target-сетью до управления маятником, и строим headline-кривую «return по ходу обучения» + greedy-оценку против случайного бейзлайна.

**Среда вместо Atari.** Ядро метода (replay, target, TD-таргет, выбор действия) не зависит от того, картинка на входе или 4 числа. Берём `CartPole-v1` — стандартный минимальный контрол-бенчмарк: те же уравнения, но обучается за полминуты на CPU, и не тонет в препроцессинге кадров. Препроцессинг φ (grayscale→84×84, стек 4 кадров) и frame-skip — это Atari-обвязка вокруг того же ядра; разберём их прозой в конце, а не кодом.

**Компьют:** CPU, ~30 c на полный прогон обучения. GPU не нужен.

**Parts:** (1) ε-greedy выбор действия → (2) TD-таргет с терминальным случаем → (3) DQN-лосс через gather → (4) сам пишешь training loop, обучаешь end-to-end → (5) headline-кривая + greedy vs random → (6) фишка: зачем нужна target-сеть.

## ⛏ Эксперименты-первыми (заполни ДО чтения метода)

Results читаем раньше abstract. В статье 2013 — таблица по 7 играм Atari (Beam Rider, Breakout, Enduro, Pong, Q*bert, Seaquest, Space Invaders), сравнение с Sarsa, Contingency и человеком; плюс графики «средний return» (шумный) против «средний максимальный предсказанный Q» (гладкий).

> **(a) Что результаты реально показали vs abstract?** На скольких из 7 игр DQN бьёт прежние методы, и на скольких — человека? Где разрыв с человеком огромный?
> …

> **(b) Самый слабый эксперимент / чем бы фальсифицировала?** Авторы говорят, что average-return-кривая шумная, и предпочитают кривую average-Q. Чем эта замена метрики удобна авторам и чем подозрительна?
> …

> **(c) Где сломалось / чего НЕ протестировали?** На каких играх DQN заметно проваливается и какое свойство этих игр он, видимо, не тянет?
> …

## Зачем это

Задача — control без модели среды: нужна оптимальная $Q^*(s,a)$ = максимальный ожидаемый дисконтированный return из состояния $s$, если сделать действие $a$ и дальше играть оптимально. Зная $Q^*$, политика тривиальна: $\arg\max_a Q^*(s,a)$.

Табличное value iteration сходится к $Q^*$, но в Atari состояний больше, чем атомов в комнате — таблицу не построить. Ставим аппроксиматор: CNN $Q(s,a;\theta)$. И тут же ломается: если в лоб подсунуть глубокой сети онлайн-поток RL, обучение разваливается. Три причины — соседние переходы сильно скоррелированы (градиенты смещены), распределение данных нестационарно (меняется вместе с политикой), и возникает положительная обратная связь (политика гонит данные туда, где сама уверена).

**Фишка статьи — experience replay.** Складываем переходы $(s,a,r,s',\text{done})$ в буфер $D$ и учимся на равномерно случайных мини-батчах из него. Это (1) рвёт корреляцию соседних шагов, (2) переиспользует каждый переход много раз (data efficiency), (3) усредняет поведенческое распределение, гася обратную связь. Альтернатива (учиться на онлайн-потоке) — ровно то, что расходится. Второй стабилизатор, добавленный в 2015, — отдельная замороженная target-сеть для таргетов; до него доберёмся в Part 6.

Почему такие данные: метод value-based и off-policy, ему нужна среда, отдающая $(s,a,r,s')$. `CartPole` даёт ровно это, без шума пиксельного препроцессинга.

## Теория (Фейнман)

**Что меряем.** Оптимальная action-value подчиняется уравнению Беллмана-оптимальности — это фиксированная точка:
$$Q^*(s,a) = \mathbb{E}_{s'}\big[\, r + \gamma\, \max_{a'} Q^*(s',a') \,\big|\, s,a \,\big].$$
Value iteration $Q_{i+1}(s,a) \leftarrow \mathbb{E}[r + \gamma \max_{a'} Q_i(s',a')]$ сходится к $Q^*$. DQN — это его сэмплированная SGD-версия: матожидание заменяем одним переходом из буфера, таблицу — сетью.

**Лосс (Eq. 2).** На итерации $i$
$$L_i(\theta_i) = \mathbb{E}_{s,a\sim\rho}\big[(y_i - Q(s,a;\theta_i))^2\big],\qquad y_i = \mathbb{E}_{s'}\big[r + \gamma \max_{a'} Q(s',a';\theta_i^-)\big].$$
Ключ: таргет $y_i$ сам зависит от весов — параметры $\theta_i^-$ держим **замороженными** на время оптимизации $\theta_i$. В 2013 это «веса предыдущей итерации», в 2015 — отдельная копия сети. Если градиент протечёт в таргет, гоняешься за собственным хвостом.

**Градиент (Eq. 3).**
$$\nabla_{\theta_i} L_i = \mathbb{E}\big[(r + \gamma \max_{a'} Q(s',a';\theta_i^-) - Q(s,a;\theta_i))\,\nabla_{\theta_i} Q(s,a;\theta_i)\big].$$

**Per-sample таргет с терминалом (Algorithm 1).** Для перехода $j$
$$y_j = \begin{cases} r_j & s'_j\ \text{терминальный}\\ r_j + \gamma \max_{a'} Q(s'_j,a';\theta^-) & \text{иначе}\end{cases}$$
и шаг по $(y_j - Q(s_j,a_j;\theta))^2$. Терминальный случай — не косметика: после конца эпизода будущего return нет, бутстрэп обязан занулиться.

**Архитектурный трюк.** Сеть берёт состояние и за один forward выдаёт вектор из $|A|$ Q-значений (по одному на действие), а не $(s,a)\to$ скаляр. Значит $\max_{a'}$ и выбор действия — это одно обращение к сети.

*Где у меня дыра в понимании:* …

❓ **Inline Question 1.** В формуле $y_i$ стоит $\theta_i^-$, а не $\theta_i$. Что конкретно сломается в обучении, если поставить $\theta_i$ и пускать градиент через таргет тоже?

**Ваш ответ:** …

In [ ]:
# --- setup (готово, просто запусти) ---
# pip install gymnasium torch numpy matplotlib
import random, math, collections, time
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import gymnasium as gym

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cpu"

class QNet(nn.Module):
    """Состояние -> вектор Q-значений, по одному на действие (single forward pass)."""
    def __init__(self, obs_dim, n_actions, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, n_actions),
        )
    def forward(self, s):
        return self.net(s)

class ReplayBuffer:
    """Буфер ёмкости N, равномерный случайный мини-батч (Eq.: D, sample uniform)."""
    def __init__(self, capacity):
        self.buf = collections.deque(maxlen=capacity)
    def push(self, s, a, r, s2, done):
        self.buf.append((s, a, r, s2, done))
    def sample(self, batch_size):
        batch = random.sample(self.buf, batch_size)
        s, a, r, s2, d = zip(*batch)
        return (np.array(s, np.float32), np.array(a, np.int64),
                np.array(r, np.float32), np.array(s2, np.float32),
                np.array(d, np.float32))
    def __len__(self):
        return len(self.buf)

def epsilon_by_step(step, eps_start=1.0, eps_end=0.05, decay=3000):
    """Экспоненциальный анил eps: 1.0 -> 0.05 (в статье 1.0 -> 0.1 линейно)."""
    return eps_end + (eps_start - eps_end) * math.exp(-step / decay)

def to_t(x):
    return torch.as_tensor(x)

_env = gym.make("CartPole-v1")
OBS_DIM = _env.observation_space.shape[0]
N_ACTIONS = _env.action_space.n
print("obs_dim", OBS_DIM, "| n_actions", N_ACTIONS)
s0, _ = _env.reset(seed=SEED)
print("sample state", np.round(s0, 3), "| QNet out shape", tuple(QNet(OBS_DIM, N_ACTIONS)(to_t(s0).unsqueeze(0)).shape))

## Part 1 — ε-greedy выбор действия

Поведенческая политика: с вероятностью $\varepsilon$ — случайное действие (exploration), иначе — жадное по Q-сети. Один forward даёт все Q сразу, бери `argmax`.

Это `rho` из лосса — то распределение, по которому реально собираются данные.

In [ ]:
def select_action(qnet, state, epsilon, n_actions):
    """ε-greedy действие по Q-сети.
    Args:
        qnet: QNet — на forward от [1, obs_dim] даёт [1, n_actions].
        state: np.ndarray [obs_dim] float — одно состояние из среды.
        epsilon: float в [0,1] — вероятность случайного действия.
        n_actions: int — число дискретных действий.
    Returns:
        int — индекс выбранного действия в [0, n_actions).
    Note: жадную ветку считай без построения графа (no_grad), это инференс.
    """
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# (1) eps=0 -> чистый argmax по Q; (2) eps=1 -> примерно равномерно по действиям.
torch.manual_seed(0); random.seed(0)
qn = QNet(OBS_DIM, N_ACTIONS)
st = np.zeros(OBS_DIM, np.float32)
with torch.no_grad():
    greedy = int(qn(to_t(st).unsqueeze(0)).argmax(1).item())
assert all(select_action(qn, st, 0.0, N_ACTIONS) == greedy for _ in range(20)), \
    "при eps=0 должен возвращаться ровно argmax Q — проверь жадную ветку"
cnt = collections.Counter(select_action(qn, st, 1.0, N_ACTIONS) for _ in range(4000))
frac = cnt[0] / 4000
print(f"Expected \u2248 0.5 доля действия 0 при eps=1 (диапазон 0.45\u20130.55) | у тебя {frac:.3f}")
assert 0.40 < frac < 0.60, "при eps=1 действия должны быть ~равновероятны — это случайная ветка"
assert all(0 <= select_action(qn, st, 0.5, N_ACTIONS) < N_ACTIONS for _ in range(50)), "индекс действия вне диапазона"
print("ok")

❓ **Inline Question 2.** ε отжигается 1.0 → 0.05, а не держится константой. Что плохого в обоих крайних режимах — ε≡1 на всём обучении и ε≡0 с самого начала?

**Ваш ответ:** …

## Part 2 — TD-таргет с терминальным случаем

Сердце метода. Для батча переходов посчитай
$$y_j = r_j + \gamma\,(1-\text{done}_j)\,\max_{a'} Q(s'_j,a';\theta^-).$$
Таргеты считает **target-сеть** и они идут под `no_grad`: градиент в них течь не должен (см. Inline Q1). Флаг `done` зануляет бутстрэп на терминальных переходах — там будущего return нет.

In [ ]:
def compute_td_targets(target_net, r, s2, done, gamma):
    """TD-таргеты y_j для батча (Algorithm 1, терминальный/нетерминальный случай).
    Args:
        target_net: QNet — замороженная копия для таргетов; forward [B, obs_dim] -> [B, n_actions].
        r: torch.FloatTensor [B] — награды r_j.
        s2: torch.FloatTensor [B, obs_dim] — следующие состояния s'_j.
        done: torch.FloatTensor [B] — 1.0 если s'_j терминальный, иначе 0.0.
        gamma: float — дисконт.
    Returns:
        torch.FloatTensor [B] — y_j; на терминальных переходах ровно r_j (бутстрэп занулён).
    Note: таргеты не должны нести градиент (no_grad / detach).
    """
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# Терминал: y == r точно. Нетерминал: y == r + gamma*max_a' Q(s2). Плюс отсутствие градиента.
torch.manual_seed(1)
tn = QNet(OBS_DIM, N_ACTIONS)
r = torch.tensor([1.0, 1.0]); s2 = torch.zeros(2, OBS_DIM); done = torch.tensor([0.0, 1.0])
y = compute_td_targets(tn, r, s2, done, 0.99)
with torch.no_grad():
    qmax = tn(s2).max(1).values
exp0 = (1.0 + 0.99 * qmax[0]).item()
print(f"Expected терминал y[1] = 1.0 (== r) | у тебя {y[1].item():.4f}")
print(f"Expected нетерминал y[0] \u2248 {exp0:.4f} | у тебя {y[0].item():.4f}")
assert abs(y[1].item() - 1.0) < 1e-6, "на терминальном переходе бутстрэп должен занулиться: y == r"
assert abs(y[0].item() - exp0) < 1e-5, "нетерминал: y = r + gamma*max_a' Q(s2); проверь max по действиям"
assert not y.requires_grad, "таргеты не должны нести градиент — оберни в no_grad/detach"
print("ok")

❓ **Inline Question 3.** В CartPole эпизод кончается и по падению маятника (term), и по достижению лимита 500 шагов (truncation). На каком из двух done должен быть 1.0 в TD-таргете, а на каком — 0.0, и почему путаница здесь систематически занижает Q?

**Ваш ответ:** …

## Part 3 — DQN-лосс через gather

Собери лосс (Eq. 2 в сэмплированной форме):
$$L = \big(y_j - Q(s_j,a_j;\theta)\big)^2.$$
Тонкость — `gather`: сеть выдаёт $[B, |A|]$, а нужен Q ровно по **сделанному** действию $a_j$, то есть $[B]$. Таргеты бери из `compute_td_targets`.

In [ ]:
def dqn_loss(qnet, target_net, s, a, r, s2, done, gamma):
    """MSE TD-лосс DQN на батче (Eq. 2/3).
    Args:
        qnet: QNet — обучаемая сеть (по ней течёт градиент).
        target_net: QNet — замороженная сеть для таргетов.
        s: [B, obs_dim] float — состояния s_j.
        a: [B] long — сделанные действия a_j (индексы).
        r: [B] float — награды; s2: [B, obs_dim] float — s'_j; done: [B] float.
        gamma: float — дисконт.
    Returns:
        torch.FloatTensor скаляр — MSE между Q(s_j,a_j;theta) и TD-таргетом y_j.
    Note: Q бери по сделанному действию a_j (gather), не max и не среднее.
    """
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# (1) форма скаляр; (2) если y и Q(s,a) совпадают -> лосс 0; (3) grad течёт в qnet, НЕ в target_net.
torch.manual_seed(2)
qn = QNet(OBS_DIM, N_ACTIONS); tn = QNet(OBS_DIM, N_ACTIONS)
B = 16
s = torch.randn(B, OBS_DIM); a = torch.randint(0, N_ACTIONS, (B,))
r = torch.randn(B); s2 = torch.randn(B, OBS_DIM); done = torch.zeros(B)
loss = dqn_loss(qn, tn, s, a, r, s2, done, 0.99)
assert loss.dim() == 0, "лосс должен быть скаляром"
loss.backward()
g_q = sum(p.grad.abs().sum() for p in qn.parameters() if p.grad is not None)
g_t = sum((p.grad is not None) for p in tn.parameters())
print(f"grad в qnet (>0 ожидаем): {g_q:.3f} | число градиентов в target_net (0 ожидаем): {g_t}")
assert g_q > 0, "градиент не дошёл до qnet — потерял граф?"
assert g_t == 0, "в target_net градиента быть не должно — таргеты под no_grad/detach"
print("ok")

## Part 4 — training loop (Algorithm 1), обучаем end-to-end

Теперь сам собери внутренний шаг главного цикла. Среда, сети, оптимизатор, буфер и синк target-сети даны. Твоя задача — тело одного шага: выбрать действие, шагнуть в среде, положить переход в буфер, и (после прогрева) сделать один градиентный шаг по `dqn_loss`.

Это и есть Algorithm 1: act → store → sample minibatch → gradient step. Обучение настоящее, ~30 c на CPU.

In [ ]:
def train_dqn(total_steps=20000, gamma=0.99, batch_size=64, lr=1e-3,
              buffer_cap=10000, target_sync=200, warmup=1000, verbose=True):
    """Главный цикл DQN (Algorithm 1). Возвращает (qnet, ep_returns).
    Тебе дано: env, qnet, target_net, opt, buf, аккумуляторы.
    Реализуй тело шага по шагам (см. # YOUR CODE).
    Args/Returns: qnet — обученная Q-сеть; ep_returns: list[float] — return за эпизод.
    Note: в буфер клади done = float(terminated) — только истинный конец зануляет бутстрэп,
          а не срез по лимиту шагов (truncation). Жадную долю в выборе действия даёт epsilon_by_step.
    """
    env = gym.make("CartPole-v1")
    obs_dim = env.observation_space.shape[0]; n_actions = env.action_space.n
    qnet = QNet(obs_dim, n_actions); target_net = QNet(obs_dim, n_actions)
    target_net.load_state_dict(qnet.state_dict())
    opt = torch.optim.Adam(qnet.parameters(), lr=lr)
    buf = ReplayBuffer(buffer_cap)

    s, _ = env.reset(seed=SEED)
    ep_ret, ep_returns = 0.0, []
    for step in range(1, total_steps + 1):
        eps = epsilon_by_step(step)
        # YOUR CODE: выбери действие a по ε-greedy (select_action), шагни в env
        #            (env.step -> s2, r, terminated, truncated, _),
        #            положи переход в буфер (done = float(terminated)),
        #            обнови s и ep_ret; на конце эпизода — запиши ep_ret в ep_returns,
        #            обнули ep_ret и сделай env.reset().
        raise NotImplementedError

        # после прогрева — один градиентный шаг по dqn_loss на случайном минибатче
        if len(buf) >= warmup:
            bs, ba, br, bs2, bd = buf.sample(batch_size)
            # YOUR CODE: посчитай loss = dqn_loss(...) на тензорах (to_t),
            #            opt.zero_grad(); loss.backward(); opt.step()
            raise NotImplementedError

        if step % target_sync == 0:
            target_net.load_state_dict(qnet.state_dict())
        if verbose and step % 4000 == 0:
            recent = np.mean(ep_returns[-20:]) if ep_returns else 0.0
            print(f"step {step:5d} eps {eps:.3f} mean20 {recent:6.1f}")
    return qnet, ep_returns

In [ ]:
# Настоящее обучение. Бейзлайн (случайная политика) на CartPole ~20. Обученная должна заметно его бить.
t0 = time.time()
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
qnet, ep_returns = train_dqn(total_steps=20000)
print("train time", round(time.time()-t0,1), "s | episodes", len(ep_returns))
last20 = float(np.mean(ep_returns[-20:]))
print(f"Expected последние-20 mean \u2248 130 (random baseline \u2248 20; порог 80) | у тебя {last20:.1f}")
assert len(ep_returns) > 30, "подозрительно мало эпизодов — крутится ли вообще цикл среды?"
assert last20 > 80, "return не оторвался от случайного бейзлайна (~20). Проверь: кладёшь ли переходы в буфер, делаешь ли opt.step, верный ли done"
print("ok")

❓ **Inline Question 4.** В буфер кладётся `done = float(terminated)`, и срез по лимиту шагов (truncation) намеренно НЕ считается терминалом. Что сломалось бы в оценках Q, если бы при truncation мы тоже ставили done=1?

**Ваш ответ:** …

## Part 5 — headline-метрика: кривая обучения + greedy vs random (готово)

Метрика задачи — return за эпизод. Строим обучающую кривую (скользящее среднее) и честно меряем **жадную** (ε=0) политику против случайного бейзлайна. Это замыкает «построил → применил → измерил той же метрикой».

In [ ]:
import matplotlib.pyplot as plt

def moving_avg(x, k=20):
    if len(x) < 1: return np.array([])
    c = np.cumsum(np.insert(np.asarray(x, float), 0, 0))
    return (c[k:] - c[:-k]) / k if len(x) >= k else np.array([np.mean(x)])

ma = moving_avg(ep_returns, 20)
plt.figure(figsize=(7,3))
plt.plot(ma)
plt.xlabel('episode'); plt.ylabel('return (скользящее ср., k=20)')
plt.title('DQN на CartPole — кривая обучения'); plt.grid(True); plt.show()

def eval_policy(qnet, epsilon, episodes=20):
    ev = gym.make('CartPole-v1'); rets = []
    for i in range(episodes):
        s,_ = ev.reset(seed=1000+i); done=False; R=0.0
        while not done:
            a = select_action(qnet, s, epsilon, N_ACTIONS)
            s,r,term,trunc,_ = ev.step(a); done = term or trunc; R += r
        rets.append(R)
    return float(np.mean(rets)), float(np.min(rets))

rnd = QNet(OBS_DIM, N_ACTIONS)
base_mean,_ = eval_policy(rnd, 1.0)      # случайная политика
dqn_mean, dqn_min = eval_policy(qnet, 0.0)  # жадная обученная
print(f'random baseline (greedy eval): mean {base_mean:.1f}')
print(f'DQN greedy: mean {dqn_mean:.1f} | min {dqn_min:.1f}')
print(f'улучшение x{dqn_mean/max(base_mean,1e-9):.1f}')

**Сравни со статьёй.** В Atari headline — таблица average-return по играм + наблюдение, что кривая **average-Q** глаже, чем кривая return. У тебя на CartPole return-кривая тоже дёрганая (vanilla DQN нестабилен), но жадная политика уверенно бьёт случайный бейзлайн (~20). Если оторвалась — ядро собрано верно.

> Твоё x-улучшение и форма кривой: …

## Part 6 — фишка: target-сеть и где она реально нужна

Замороженная target-сеть (Nature 2015) — второй стабилизатор поверх replay: таргеты считает копия $\theta^-$, которую синкаем редко (раз в C шагов), а не текущие веса. Идея — не дать таргету гоняться за собственным движением.

Проверим вклад честным контрастом: то же обучение, но target-сеть синкается **каждый шаг** (`target_sync=1`) — это эквивалент «без отдельной target-сети, таргеты по текущим весам» — против редкого синка раз в C шагов. Никаких assert на «кто выиграл»: просто запусти обе ветви на нескольких сидах и посмотри на кривые и разброс. Результат тут поучителен сам по себе (см. вопрос ниже).

In [ ]:
def run_seeds(target_sync, seeds=(0,1,2), total_steps=20000):
    finals = []
    curves = []
    for sd in seeds:
        torch.manual_seed(sd); np.random.seed(sd); random.seed(sd)
        qn, rets = train_dqn(total_steps=total_steps, target_sync=target_sync, verbose=False)
        m,_ = eval_policy(qn, 0.0)
        finals.append(m); curves.append(moving_avg(rets,20))
    return finals, curves

fin_C, cur_C = run_seeds(200)   # редкий синк (frozen target)
fin_1, cur_1 = run_seeds(1)     # синк каждый шаг (~без target-сети)

plt.figure(figsize=(7,3))
for c in cur_C: plt.plot(c, color='C0', alpha=0.6)
for c in cur_1: plt.plot(c, color='C1', alpha=0.6)
plt.plot([],[],color='C0',label='sync=200 (frozen target)')
plt.plot([],[],color='C1',label='sync=1 (~без target-сети)')
plt.xlabel('episode'); plt.ylabel('return (ma20)'); plt.legend(); plt.grid(True)
plt.title('target-сеть: разброс по сидам'); plt.show()
print(f'greedy eval | sync=200: {[round(x) for x in fin_C]} mean {np.mean(fin_C):.0f}')
print(f'greedy eval | sync=1  : {[round(x) for x in fin_1]} mean {np.mean(fin_1):.0f}')

**Что ты, скорее всего, увидела.** На CartPole редкий синк target-сети НЕ даёт явного выигрыша — нередко синк-каждый-шаг учится даже быстрее. Это не баг твоей реализации: задача низкоразмерная, награда плотная, аппроксиматор маленький — те расходящиеся обратные связи, которые target-сеть гасит, тут почти не бьют. Выигрыш от заморозки таргета проявляется в **тяжёлом** режиме — высокоразмерный вход (пиксели), глубокая CNN, разреженная награда, — то есть ровно в Atari, где её и ввели. Мораль ШАД: «стабилизатор» доказывают на том режиме, где есть нестабильность; на лёгком бенчмарке он может быть невидимым или даже мешать. Не подгоняй вывод под лозунг — читай, что показали сиды.

**Что важно про источник.** Тема узла называет target-сеть, но в самой статье 2013 (arXiv:1312.5602) **отдельной замороженной target-сети нет** — там таргеты пишутся через $\theta_{i-1}$ (веса предыдущей итерации, держатся фиксированными на время оптимизации лосса) плюс experience replay. Явная периодически синкаемая $Q(\cdot;\theta^-)$ появилась только в Nature 2015 (*Human-level control through deep RL*). Мы реализовали полный DQN (replay 2013 + target 2015), потому что узел про него; это два разных вклада из двух статей. Double DQN и prioritized replay — ещё позже; в 2013 prioritized sweeping упомянут лишь как future work.

❓ **Inline Question 5.** По твоим сидам редкий синк не выиграл. Сформулируй, какие три свойства Atari (vs CartPole) делают замороженную target-сеть необходимой — и придумай, как бы ты на CartPole искусственно создала режим, где заморозка начнёт помогать.

**Ваш ответ:** …

## Препроцессинг φ и Atari-обвязка (прозой, без кода)

Чего мы НЕ кодили, но что нужно для пиксельного DQN из статьи:
- **φ:** grayscale + downsample/crop до 84×84, и **стек последних 4 кадров** → состояние фиксированной длины. Стек кадров чинит частичную наблюдаемость (по одному кадру не видно скорость/направление).
- **Frame-skip k=4:** агент выбирает действие раз в 4 кадра, между ними оно повторяется — дешевле и не вредит.
- **Reward clipping в {-1,0,+1}:** уравнивает масштаб наград между играми, чтобы один learning rate работал везде.
- **Архитектура 2013:** вход 84×84×4 → conv 16×8×8 stride4 ReLU → conv 32×4×4 stride2 ReLU → FC 256 ReLU → линейный выход по числу действий; RMSProp, minibatch 32, replay 1M кадров.

Ядро (replay, target, TD-таргет, ε-greedy, gather-лосс) — ровно то, что ты собрала. Свап CartPole→Atari меняет только энкодер состояния и эту обвязку.

## Рефлексия

**Что украсть в свою работу:** replay-буфер + замороженная target-копия — общий приём против нестационарных самопорождаемых таргетов, не только в RL. Где ещё цель обучения зависит от текущих весов и потому уплывает?
> …

**В какой НОВОЙ ситуации DQN — правильный инструмент:** дискретные действия, off-policy, доступен поток $(s,a,r,s')$ и хочется переиспользовать опыт. Где ломается: непрерывные/огромные пространства действий (там $\max_{a'}$ нетривиален → DDPG/SAC).
> …

**Что осталось непонятным → следующий узел beam:** напр. Double DQN (overestimation от $\max$), prioritized replay (неравномерный сэмплинг), dueling-архитектура.
> …